# Урок 5 — Списки, кортежі та множини

> Сценарій: до нас прийшов власник ресторану з реальними чеками. Потрібно зберегти дані про кожен чек так, щоб з ними було зручно й безпечно працювати — і не переплутати, де сума рахунку, а де чайові.

Структура уроку: **RETRIEVE → CONCEPT → CREATE → TRANSFER** (та сама послідовність, що й в Уроках 3, 4, 6, 7).

## 🔁 RETRIEVE — пригадай Уроки 3–4 (без підглядання)

1. Чим `str` відрізняється від `int`/`float` з точки зору того, що з ним можна робити?
2. Що поверне `5 > 3 and 2 > 10`?
3. Навіщо взагалі потрібен `while`, якщо вже є `for`? (поки що можна відповісти інтуїтивно — детально розберемо в Уроці 6)

<details>
<summary>Відповіді</summary>

1. <code>str</code> — послідовність символів (індексація, конкатенація через <code>+</code>, зрізи); <code>int</code>/<code>float</code> — числа (арифметика). Спроба <code>"5" + 3</code> — <code>TypeError</code>.
2. <code>False</code> — <code>2 > 10</code> хибне, а <code>and</code> вимагає істинності обох частин.
3. <code>while</code> повторює, поки виконується умова, і наперед невідомо скільки разів — на відміну від <code>for</code>, який іде по вже готовій колекції відомого розміру.

</details>

## 📖 CONCEPT

### 1. Один чек — і проблема звичайного кортежу

Уявімо, що кожен чек ресторану ми зберігаємо як `tuple`: сума рахунку, чайові, стать клієнта, чи курець, день тижня, час (обід/вечеря), кількість осіб. Кортеж підходить ідеально — чек уже виписаний, це **факт, що стався**, і його не можна (і не треба) міняти. Але подивимось, що стається, коли до такого чека треба звернутись:

In [1]:
# Один чек — звичайний tuple
raw_order = (23.68, 3.31, "Male", "No", "Sun", "Dinner", 2)

print(raw_order)
print()

# Що таке raw_order[0]? raw_order[3]? Доводиться постійно пам'ятати порядок полів
print("Сума чеку?", raw_order[0])   # 0 = total_bill — треба пам'ятати
print("День?     ", raw_order[4])   # 4 = day
print("Чайові?   ", raw_order[1])   # 1 = tip

# Через тиждень, повернувшись до цього коду, доведеться згадувати що є що —
# а якщо переплутати індекс — код мовчки порахує щось не те, без жодної помилки

(23.68, 3.31, 'Male', 'No', 'Sun', 'Dinner', 2)

Сума чеку? 23.68
День?      Sun
Чайові?    3.31


### 2. `NamedTuple` — той самий кортеж, але з іменами полів

`NamedTuple` вирішує проблему індексів: кожне поле отримує ім'я, залишаючись при цьому звичайним, незмінним кортежем — той самий розмір у пам'яті, та сама швидкість, та сама підтримка розпакування й індексації.

In [2]:
from typing import NamedTuple


class Order(NamedTuple):
    total_bill: float
    tip: float
    sex: str
    smoker: str
    day: str
    time: str
    size: int


order = Order(
    total_bill=23.68,
    tip=3.31,
    sex="Male",
    smoker="No",
    day="Sun",
    time="Dinner",
    size=2,
)

print(order)
print()
print(f"Через ім'я:   order.total_bill = {order.total_bill}")
print(f"Через індекс: order[0]         = {order[0]}   ← те саме значення")
print()

# NamedTuple — це справжній tuple
print("isinstance(order, tuple):", isinstance(order, tuple))

# Розпакування працює так само, як у звичайного tuple
total_bill, tip, sex, smoker, day, time, size = order
print(f"Розпакування: сума={total_bill}, чайові={tip}, день={day}")
print()

# Незмінність — захист від випадкової зміни
try:
    order.tip = 5.0
except AttributeError as e:
    print(f"AttributeError: {e}")
    print("NamedTuple — незмінний (immutable), як і звичайний tuple")

Order(total_bill=23.68, tip=3.31, sex='Male', smoker='No', day='Sun', time='Dinner', size=2)

Через ім'я:   order.total_bill = 23.68
Через індекс: order[0]         = 23.68   ← те саме значення

isinstance(order, tuple): True
Розпакування: сума=23.68, чайові=3.31, день=Sun

AttributeError: can't set attribute
NamedTuple — незмінний (immutable), як і звичайний tuple


### 3. Три контейнери, три призначення

| | `list` | `tuple` | `set` |
|---|---|---|---|
| Змінність | змінний | незмінний | змінний |
| Порядок | зберігає | зберігає | не гарантує |
| Дублікати | дозволені | дозволені | видаляються автоматично |
| Коли обирати | послідовність, що росте/змінюється по ходу програми (список усіх чеків) | фіксований запис, що вже стався (один чек) | потрібна лише унікальність і швидка перевірка належності (унікальні дні роботи) |

Правило вибору не про синтаксис — про **сенс**: чи ця колекція буде змінюватись, чи в ній важливий порядок, чи важлива лише унікальність.

## 🛠️ CREATE — від кількох чеків до реального датасету

Почнемо з невеликого, вручну введеного набору чеків — щоб побачити механіку без зайвого шуму. Потім застосуємо **той самий код, без жодної зміни**, до реального датасету на 244 чеки. Це і є головна ідея уроку: алгоритм не знає й не піклується, звідки прийшли дані.

### Крок 1 — невеликий вручну введений набір

П'ять реальних на вигляд чеків, кожен — `Order`:

In [3]:
orders_manual = [
    Order(23.68,  3.31, "Male",   "No",  "Sun",  "Dinner", 2),
    Order(11.61,  1.50, "Male",   "No",  "Sat",  "Dinner", 2),
    Order(15.06,  3.00, "Female", "Yes", "Sat",  "Dinner", 3),
    Order(10.34,  1.66, "Male",   "No",  "Sun",  "Lunch",  3),
    Order(24.59,  3.61, "Female", "No",  "Fri",  "Lunch",  4),
]

print(f"Чеків у наборі: {len(orders_manual)}")
for o in orders_manual:
    tip_pct = o.tip / o.total_bill * 100
    print(f"  {o.day:4} {o.time:7} ${o.total_bill:6.2f}  tip {tip_pct:4.1f}%  {o.size} ос.")

Чеків у наборі: 5
  Sun  Dinner  $ 23.68  tip 14.0%  2 ос.
  Sat  Dinner  $ 11.61  tip 12.9%  2 ос.
  Sat  Dinner  $ 15.06  tip 19.9%  3 ос.
  Sun  Lunch   $ 10.34  tip 16.1%  3 ос.
  Fri  Lunch   $ 24.59  tip 14.7%  4 ос.


### Крок 2 — той самий алгоритм, реальний датасет

Датасет `tips` (вбудований у `seaborn`) містить **244 реальних чеки** з американського ресторану — ті самі 7 полів, що і в нашому `Order`. DataFrame — таблиця `pandas`; наш алгоритм працює зі списком `Order`, тож спершу конвертуємо. Спочатку звичайним циклом — щоб побачити кожен крок:

In [4]:
import seaborn as sns

tips_df = sns.load_dataset("tips")
print(f"Завантажено: {len(tips_df)} рядків, колонки: {list(tips_df.columns)}")
print()

# Варіант 1 — for loop з append (зрозуміло крок за кроком)
orders = []

for _, row in tips_df.iterrows():
    order = Order(
        total_bill=float(row["total_bill"]),
        tip=float(row["tip"]),
        sex=str(row["sex"]),
        smoker=str(row["smoker"]),
        day=str(row["day"]),
        time=str(row["time"]),
        size=int(row["size"]),
    )
    orders.append(order)

print(f"Тип:       {type(orders).__name__}")
print(f"Елемент:   {type(orders[0]).__name__}")
print(f"Кількість: {len(orders)}")
print(f"Перший чек: {orders[0]}")

Завантажено: 244 рядків, колонки: ['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size']

Тип:       list
Елемент:   Order
Кількість: 244
Перший чек: Order(total_bill=16.99, tip=1.01, sex='Female', smoker='No', day='Sun', time='Dinner', size=2)


Той самий результат — list comprehension, компактніше (comprehensions докладно розбираємо в Уроці 6; тут — лише як зручніша форма запису вже зрозумілого циклу):

In [5]:
orders_v2 = [
    Order(
        total_bill=float(row["total_bill"]),
        tip=float(row["tip"]),
        sex=str(row["sex"]),
        smoker=str(row["smoker"]),
        day=str(row["day"]),
        time=str(row["time"]),
        size=int(row["size"]),
    )
    for _, row in tips_df.iterrows()
]

assert orders_v2 == orders
print("Той самий результат, що й циклом вище")
print()
print("Головний висновок: код, який ми написали для 5 вручну введених чеків,")
print("працює БЕЗ ЖОДНОЇ ЗМІНИ і на 244 реальних чеках —")
print("алгоритм не знає і не піклується, звідки прийшли дані.")

Той самий результат, що й циклом вище

Головний висновок: код, який ми написали для 5 вручну введених чеків,
працює БЕЗ ЖОДНОЇ ЗМІНИ і на 244 реальних чеках —
алгоритм не знає і не піклується, звідки прийшли дані.


### Крок 3 — `set`: скільки в нас унікальних днів і часів

Датасет охоплює кілька днів тижня і два прийоми їжі. Скільки саме — рахувати вручну не потрібно, `set` сам прибере дублікати:

In [6]:
unique_days = set()
for o in orders:
    unique_days.add(o.day)   # дублікат — просто ігнорується

unique_times = {o.time for o in orders}   # set comprehension — той самий принцип

print("Дні роботи:", unique_days)
print("Час:       ", unique_times)
print()
print(f"Усього чеків: {len(orders)}, але лише {len(unique_days)} унікальних днів")

Дні роботи: {'Fri', 'Thur', 'Sun', 'Sat'}
Час:        {'Dinner', 'Lunch'}

Усього чеків: 244, але лише 4 унікальних днів


### Крок 4 — `Counter`: скільки чеків на кожен день

`collections.Counter` — спеціалізований словник для підрахунку: скільки разів кожне значення зустрілось. Для «скільки чеків у кожен день» — рахувати вручну не потрібно:

In [7]:
from collections import Counter

orders_per_day = Counter(o.day for o in orders)

print("Чеків по днях:")
for day, count in orders_per_day.most_common():
    print(f"  {day:4}: {count} чеків")

print()
print(f"Найзавантаженіший день: {orders_per_day.most_common(1)[0][0]}")

assert sum(orders_per_day.values()) == len(orders)
print("OK — сума по днях збігається із загальною кількістю чеків")

Чеків по днях:
  Sat : 87 чеків
  Sun : 76 чеків
  Thur: 62 чеків
  Fri : 19 чеків

Найзавантаженіший день: Sat
OK — сума по днях збігається із загальною кількістю чеків


## 🔄 TRANSFER — самостійні задачі

Той самий `orders: list[Order]`, дві незалежні задачі на фільтрацію та агрегацію — тільки `list`/`tuple`/`set`, без `dict` (той інструмент — тема Уроку 6).

### Задача 1 — аналітика вечері (Dinner)

Знайди всі чеки з `time == "Dinner"`, порахуй середній чек і середній % чайових серед них.

In [8]:
def dinner_report(orders: list) -> tuple:
    """Повертає (кількість вечірніх чеків, середній чек, середній tip%)."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    dinner_orders = [o for o in orders if o.time == "Dinner"]

    avg_bill = sum(o.total_bill for o in dinner_orders) / len(dinner_orders)
    avg_tip_pct = sum(o.tip / o.total_bill * 100 for o in dinner_orders) / len(dinner_orders)

    return len(dinner_orders), avg_bill, avg_tip_pct
    # END SOLUTION


count, avg_bill, avg_tip_pct = dinner_report(orders)
print(f"Вечірніх чеків: {count}")
print(f"Середній чек:   ${avg_bill:.2f}")
print(f"Середній tip%:  {avg_tip_pct:.1f}%")

assert count == 176
assert round(avg_bill, 2) == 20.80
print("OK")

Вечірніх чеків: 176
Середній чек:   $20.80
Середній tip%:  16.0%
OK


### Задача 2 — великі столи

Знайди всі чеки, де `size >= 4`, порахуй середній чек і середній чек **на одну людину**.

In [9]:
def large_tables_report(orders: list) -> tuple:
    """Повертає (кількість великих столів, середній чек, середній чек на людину)."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    large_tables = [o for o in orders if o.size >= 4]

    avg_bill = sum(o.total_bill for o in large_tables) / len(large_tables)
    avg_per_person = sum(o.total_bill / o.size for o in large_tables) / len(large_tables)

    return len(large_tables), avg_bill, avg_per_person
    # END SOLUTION


count, avg_bill, avg_pp = large_tables_report(orders)
print(f"Великих столів (4+):    {count}")
print(f"Середній чек:           ${avg_bill:.2f}")
print(f"Середній чек на людину: ${avg_pp:.2f}")

assert count == 46
print("OK")

Великих столів (4+):    46
Середній чек:           $29.31
Середній чек на людину: $6.91
OK


## ✅ Самоперевірка (5 запитань)

**1.** Чим `NamedTuple` відрізняється від звичайного `dict` з тими самими ключами-іменами?

<details><summary>Відповідь</summary><code>NamedTuple</code> — незмінний (immutable), фіксований набір полів, підтримує індексацію й розпакування як звичайний <code>tuple</code>. <code>dict</code> — змінний, ключі можна додавати/видаляти в будь-який момент, розпакування за позицією не працює так само напряму.</details>

**2.** Чому для одного чека обрали `tuple`/`NamedTuple`, а не `list`?

<details><summary>Відповідь</summary>Чек — уже здійснена подія, факт. <code>tuple</code> захищає його від випадкової зміни; <code>list</code> дозволяв би випадково змінити суму рахунку заднім числом.</details>

**3.** `{}` — це порожній `set` чи порожній `dict`?

<details><summary>Відповідь</summary><code>dict</code>. Порожня множина — лише <code>set()</code>.</details>

**4.** Що конкретно демонструє асерт `orders_v2 == orders` (Крок 2)?

<details><summary>Відповідь</summary>Що <code>for</code>-цикл з <code>append</code> і list comprehension дають абсолютно однаковий результат — це той самий алгоритм, лише інша форма запису.</details>

**5.** Навіщо тут `Counter`, якщо те саме можна порахувати циклом і словником `d[key] = d.get(key, 0) + 1`?

<details><summary>Відповідь</summary><code>Counter</code> робить рівно те саме, але без ручного написання циклу для підрахунку — і одразу дає зручні методи на кшталт <code>.most_common()</code>. Детальний розбір патерну підрахунку через звичайний <code>dict</code> — тема Уроку 6.</details>

### Шпаргалка

```python
# NamedTuple — tuple з іменами полів
from typing import NamedTuple

class Order(NamedTuple):
    total_bill: float
    tip: float

order = Order(23.68, 3.31)
order.total_bill      # замість order[0]
isinstance(order, tuple)   # True — це досі tuple
order.tip = 5.0        # AttributeError — незмінний

# set — унікальність без ручної перевірки
unique_days = {o.day for o in orders}   # set comprehension

# Counter — підрахунок входжень
from collections import Counter
Counter(o.day for o in orders).most_common()
```

## Далі

Урок 6 («Словники, for, comprehensions») продовжує напряму з того самого `orders: list[Order]`, який ми щойно побудували — тепер ці 244 чеки агрегуються по днях і часу доби через `dict`, з'являється `defaultdict`, а comprehensions розбираються докладно.